In [0]:
!pip install -U -qqqq backoff databricks-openai uv databricks-agents mlflow-skinny[databricks] databricks_langchain
!pip install -U databricks-langchain langchain-community langchain 
!pip install unitycatalog-langchain[databricks]
!pip install unitycatalog-ai[databricks]
!pip install -U databricks-sdk

# Restart the Python VM so the environment picks up the new packages
%restart_python

In [0]:
%run ../../Includes/_common

In [0]:
# Create a python DA object from the dbacademy.ops.meta table
DA = DBAcademyHelper()
DA.init()

In [0]:
def use_uc_env():
    catalog_name = DA.catalog_name
    schema_name = DA.schema_name 

    spark.sql(f"USE CATALOG {DA.catalog_name}")
    spark.sql(f"USE SCHEMA {DA.schema_name}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS agent_vol")
    return catalog_name, schema_name 

In [0]:
# Set UC environment
catalog_name, schema_name = use_uc_env()

In [0]:
def set_environment_and_tools(catalog_name:str, schema_name:str) -> None:
    """
    Sets the environment and tools for the notebook.
    """

    query1 = f"""
    USE CATALOG {catalog_name}
    """
    query2 = f"""
    USE SCHEMA {schema_name}
    """
    query3 = """
    DROP FUNCTION IF EXISTS avg_neigh_price
    """
    query4 = """
    DROP FUNCTION IF EXISTS cnt_by_room_type
    """
    query5 = f"""
    CREATE OR REPLACE FUNCTION avg_neigh_price(
    neighborhood_name STRING COMMENT "The neighborhood name to filter by (e.g., 'Mission', 'Upper Market')"
    )
    RETURNS DOUBLE
    LANGUAGE SQL
    DETERMINISTIC
    COMMENT 'Calculates the average listing price for a specific neighborhood in San Francisco. Returns the average price as a numeric value. Price strings are cleaned and converted to numeric values before averaging.'
    RETURN 
    SELECT AVG(CAST(REGEXP_REPLACE(price, '[^0-9.]', '') AS DOUBLE))
    FROM sf_airbnb_listings
    WHERE neighbourhood_cleansed = neighborhood_name
    AND price IS NOT NULL
    AND REGEXP_REPLACE(price, '[^0-9.]', '') != ''
    """
    query6 = f"""
    CREATE OR REPLACE FUNCTION cnt_by_room_type(
    neighborhood_name STRING COMMENT "The neighborhood name to filter by",
    room_type_filter STRING COMMENT "The room type to count (e.g., 'Private room' or 'Shared room')"
    )
    RETURNS BIGINT
    LANGUAGE SQL
    DETERMINISTIC
    COMMENT 'Counts the number of Airbnb listings for a specific room type in a given neighborhood. Returns the count as an integer.'
    RETURN
    SELECT COUNT(*)
    FROM sf_airbnb_listings
    WHERE neighbourhood_cleansed = neighborhood_name
        AND room_type = room_type_filter
    """
    print(f"Using catalog `{catalog_name}` and schema `{schema_name}`")
    spark.sql(query1).collect()
    spark.sql(query2).collect()
    
    print("Creating functions...")
    spark.sql(query3).collect()
    spark.sql(query4).collect()
    
    spark.sql(query5).collect()
    print(f"Created function avg_price_by_neighborhood(neighborhood_name STRING)")
    
    spark.sql(query6).collect()
    print(f"Created function cnt_by_room_type(neighborhood_name STRING)")
    return None

In [0]:
# create json file demo_agent1.json

import json

def create_demo_agent_config(catalog_name, schema_name, output_path="./demo_agent1_config.json") -> None:
    """
    Creates a JSON configuration file for demo_agent1.
    
    Args:
        output_path (str): Path where the JSON file will be saved. 
                          Defaults to "demo_agent1_config.json"
    
    Returns:
        str: Path to the created file
    """
    config = {
        "llm_endpoint": "databricks-gpt-oss-120b",
        "llm_temperature": 0.1,
        "system_prompt": "You are a helpful assistant. Make sure to use tools for additional functionality.",
        "tool_list": [
            f"{catalog_name}.{schema_name}.avg_neigh_price",
            f"{catalog_name}.{schema_name}.cnt_by_room_type"
        ]
    }
    
    with open(output_path, 'w') as f:
        json.dump(config, f, indent=4)
    
    print(f"Configuration file created successfully at: {output_path}")
    return None

In [0]:
# Set the environment
set_environment_and_tools(catalog_name, schema_name)

# Create the agent's configuration file
create_demo_agent_config(catalog_name, schema_name)